# 05 · Is the shape parameter constant across thresholds? (Wald test)

**Hypotheses**

* H0: k₁ = k₂ = … = kₘ — the same shape at every threshold.
* H1: at least one threshold has a different shape.

**Method**

1. Fit the chosen model (Gamma or Weibull) separately at each threshold →
   shape estimates k̂₁ … k̂ₘ.
2. Estimate the covariance matrix Σ of those estimates with a bootstrap:
   resample the catalogue rows with replacement, rebuild the waiting times
   and refit every threshold. The thresholds are nested (the same bursts
   appear in several of them), so the estimates are correlated and Σ is far
   from diagonal.
3. Turn the m shapes into m−1 differences with respect to the first one,
   d = C·k̂, and compute

   W = dᵀ (C Σ Cᵀ)⁻¹ d,  which under H0 follows a χ² with m−1 degrees of freedom.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd

from srb.config import RANDOM_SEED
from srb.data import inter_event_times_by_threshold, load_catalogue, select_thresholds
from srb.distributions import DISTRIBUTIONS
from srb.fitting import fit_mle
from srb.wald import bootstrap_shape_covariance, wald_test_equal_shapes

## Parameters

In [2]:
MODEL = "gamma"  # "gamma" or "weibull"
ALPHA = 0.05  # significance level
START_THRESHOLD = 4000  # SFU
END_THRESHOLD = 100000  # SFU
N_BOOTSTRAP = 10000

## 1. Per-threshold fits on the original data

In [3]:
dist = DISTRIBUTIONS[MODEL]
df = load_catalogue()
groups = inter_event_times_by_threshold(df, select_thresholds(START_THRESHOLD, END_THRESHOLD))
if len(groups) < 2:
    raise RuntimeError("Need at least two valid thresholds to test equality of shapes.")

fits = {threshold: fit_mle(times, dist) for threshold, times in groups.items()}
if not all(fit.success for fit in fits.values()):
    raise RuntimeError("At least one original fit failed.")

thresholds = np.array(list(fits))
shape_estimates = np.array([fit.params[0] for fit in fits.values()])

Skipping T = 100000.00: only 6 inter-event times


## 2. Bootstrap covariance of the shape estimates

This is the slow step (one full refit of every threshold per bootstrap
sample).

In [4]:
rng = np.random.default_rng(RANDOM_SEED)
shape_covariance, shape_bootstrap = bootstrap_shape_covariance(df, thresholds, dist, N_BOOTSTRAP, rng)
print(f"Bootstrap fits used: {len(shape_bootstrap)} / {N_BOOTSTRAP}")

Bootstrap fits used: 329 / 10000


In [5]:
results = pd.DataFrame({
    "threshold": thresholds,
    "N": [len(times) for times in groups.values()],
    "k_hat": shape_estimates,
    "boot_se_k_hat": np.sqrt(np.maximum(np.diag(shape_covariance), 0.0)),
    "scale_hat": [fit.params[-1] for fit in fits.values()],
})
results

,threshold,N,k_hat,boot_se_k_hat,scale_hat
0,4281.332399,183,0.244893,0.019292,478.961217
1,5455.594781,161,0.235532,0.019905,566.046858
2,6951.927962,128,0.236047,0.023249,710.396218
3,8858.667904,103,0.232713,0.024043,891.020605
4,11288.378917,84,0.243921,0.027731,1042.304682
5,14384.498883,67,0.258983,0.039034,1230.853642
6,18329.807108,54,0.298827,0.050901,1323.501102
7,23357.214691,45,0.330057,0.054199,1437.912072
8,29763.514416,34,0.334762,0.061821,1876.335530
9,37926.901907,31,0.320144,0.064108,2151.919661


In [6]:
print("Bootstrap covariance matrix used for k_hat:")
print(shape_covariance)

Bootstrap covariance matrix used for k_hat:
[[0.0003722  0.00037103 0.0003767  0.00030887 0.00031648 0.0003676
  0.0003339  0.00023686 0.00026811 0.00027519 0.00010537 0.00019682
  0.00026821]
 [0.00037103 0.00039621 0.00040503 0.00033717 0.00033875 0.00039177
  0.00035294 0.00024006 0.00027656 0.00027886 0.00013829 0.0002368
  0.00029529]
 [0.0003767  0.00040503 0.00054051 0.00045131 0.00045377 0.00050405
  0.00043179 0.00028285 0.00035072 0.00035284 0.00016506 0.0002801
  0.00036327]
 [0.00030887 0.00033717 0.00045131 0.00057807 0.00056015 0.00060265
  0.00057015 0.00035784 0.00047404 0.00048505 0.00027382 0.00031451
  0.00034257]
 [0.00031648 0.00033875 0.00045377 0.00056015 0.00076902 0.00087616
  0.00085192 0.00060798 0.00070258 0.00070976 0.0004879  0.00053067
  0.00051226]
 [0.0003676  0.00039177 0.00050405 0.00060265 0.00087616 0.00152368
  0.00142902 0.00107815 0.0011647  0.00118359 0.00079967 0.00073115
  0.00072428]
 [0.0003339  0.00035294 0.00043179 0.00057015 0.00085192 0.

## 3. Wald test

In [7]:
W, dof, p_value = wald_test_equal_shapes(shape_estimates, shape_covariance)

print(f"Model: {MODEL}")
print("H0: k_1 = ... = k_m")
print("H1: at least one threshold has a different k\n")
print(f"Wald statistic = {W:.6f}")
print(f"df             = {dof}")
print(f"p-value        = {p_value:.6g}\n")

if p_value < ALPHA:
    print(f"Decision at alpha={ALPHA}: reject H0; shape is not constant.")
else:
    print(f"Decision at alpha={ALPHA}: do not reject H0; no evidence shape changes.")

Model: gamma
H0: k_1 = ... = k_m
H1: at least one threshold has a different k

Wald statistic = 23.030316
df             = 12
p-value        = 0.0274697

Decision at alpha=0.05: reject H0; shape is not constant.
